## 1. Imports

In [1]:
import sys

# Strip any cached bouquet paths so this notebook always runs against
# the bouquet_coil_bounds worktree.  Restart the kernel after editing.
sys.path[:] = [p for p in sys.path
               if not p.endswith('/Users/danielburgess/Desktop/plasma/bouquet')
               and not p.endswith('/Users/danielburgess/Desktop/plasma/bouquet_phantom_vsc')
               and not p.endswith('/Users/danielburgess/Desktop/plasma/bouquet_tier3')]
sys.path.insert(0, '/Users/danielburgess/Desktop/plasma/bouquet_coil_bounds')

for mod in list(sys.modules):
    if mod == 'bouquet' or mod.startswith('bouquet.'):
        del sys.modules[mod]

import bouquet
assert 'bouquet_coil_bounds' in bouquet.__file__, \
    f"wrong bouquet: {bouquet.__file__}"
import inspect
assert 'coil_drift' in str(inspect.signature(bouquet.generate_bouquet)), \
    "coil_drift kwarg missing -- pull latest"
print("OK -", bouquet.__file__)


OK - /Users/danielburgess/Desktop/plasma/bouquet_coil_bounds/bouquet/__init__.py


In [2]:
import os
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d
from omfit_classes.omfit_nc import OMFITnc

%matplotlib inline
%config InlineBackend.figure_format = "retina"

# --- TokaMaker ---
#sys.path.append('/Applications/OpenFUSIONToolkit/python')
### FIX THIS
tokamaker_python_path = '/Users/danielburgess/Desktop/plasma/OpenFUSIONToolkit/build_release'
if tokamaker_python_path is not None:
    sys.path.append(os.path.join(tokamaker_python_path, 'python'))

from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import create_power_flux_fun

# --- Bouquet ---
from bouquet import (
    read_geqdsk,
    reconstruct_equilibrium,
    generate_bouquet,
    initialize_equilibrium_database,
    store_equilibrium,
    calc_cylindrical_li_proxy,
    plot_tokamaker_comparison,
    plot_bouquet,plot_pfile_bouquet,plot_traces
)

from bouquet import PFile  
from bouquet.io.pfile import read_pfile
from bouquet import new_uncertainty_profiles
from bouquet import sigmoid_length_scale
from bouquet import read_eqdsk_from_bytes, load_equilibrium


print(f"Working directory: {os.getcwd()}")

Could not collect arrays: ValueError('setting an array element with a sequence.')
Working directory: /Users/danielburgess/Desktop/plasma/bouquet_coil_bounds/examples


## 2. User configuration

All tunable parameters are collected here.  Key choices:

- **Uncertainties for $n_e$, $T_e$, $T_i$** are computed from raw data
  (see §3) rather than using a fixed fractional envelope.
- **$j_\phi$ uncertainty** uses a flat 10% fractional envelope (standard).
- **GPR length scales** are kept at standard D3D values.
- **`n_equils = 15`** perturbed equilibria to generate.

In [3]:
# ----- Input files -----
geqdsk_file = 'g204441.04409_719'
IDA_file  = 'IDA_204441_.cdf'

header = 'g204441.4400_results'  # HDF5 database name (without .h5)
gpsave_header = '204441.4400'

sim_time = 4409 # only used for workflow_num = 2 (needs to be standardized later)

# ----- DIII-D machine files -----
MESH_FILE = '/Users/danielburgess/Desktop/plasma/SPARC/8.7T/Priyansh_199749_q4/DIIID_mesh.h5'#'DIIID_mesh.h5'   # Pre-built TokaMaker mesh for DIII-D

# ----- j_phi uncertainty (flat fractional envelope — kept as-is) -----
frac_jphi = 0.10

# ----- GPR correlation length scales (in psi_N units — kept as-is) -----
n_ls = 0.6    # density
t_ls = 0.5    # temperature
j_ls = 0.3    # current density

# ----- Perturbation settings -----
n_equils = 15        # number of perturbed equilibria
jBS_scale_range = [0.99, 1.01]  # bootstrap current multiplicative range
pad_psi  = 1e-3      # LCFS psi padding for TokaMaker queries

# ----- Ion species for p-file (from p-file footer) -----
ion_N = [6, 1, 1]
ion_Z = [6, 1, 1]
ion_A = [12, 2, 2]

# ----- Which posterior summary to use for the uncertainty band -----
# 'std'         → mean ± 1 std  (Gaussian-like summary)
# 'percentile'  → median + 16th/84th percentile band (robust to outliers)
UNCERTAINTY_METHOD = 'percentile'   # change to 'std' if preferred

# ----- Output directory for figures -----
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Will generate {n_equils} perturbed equilibria")
print(f"Figures will be saved to: {FIG_DIR}/")

Will generate 15 perturbed equilibria
Figures will be saved to: figures/


## 3. Compute 1σ uncertainties from raw profile data

The file `profiles_data.npz` contains raw measurement data for $n_e$,
$T_e$, and $T_i$ (e.g. from Thomson scattering and charge-exchange
recombination spectroscopy).  These are scattered measurements at
various $\psi_N$ locations.

The p-file contains the **fitted** (smooth) profiles.  The approach:

1. Interpolate the fitted p-file profile onto the raw measurement $\psi_N$ grid.
2. Compute **residuals** = raw data − fitted profile.
3. Compute a **rolling standard deviation** of residuals (window size
   controlled by `SMOOTH_WINDOW`) to get a smooth, radially-varying
   $\sigma(\psi_N)$.
4. Interpolate this smooth $\sigma(\psi_N)$ onto the g-file's uniform
   $\psi_N$ grid for use in bouquet.

This captures the actual measurement scatter rather than assuming a
fixed fractional uncertainty, giving a more faithful representation
of where the profiles are well-constrained vs. uncertain.

**Note:** We only use raw data for $n_e$, $T_e$, $T_i$.  The $n_i$
uncertainty is derived from $n_e$ (via quasi-neutrality), and the
$j_\phi$ uncertainty uses a standard flat 10% envelope.

In [4]:
# ---- Load IDA posterior samples ----
cdf = OMFITnc(IDA_file)

pf_time = np.array(cdf['time']['data']) / 1e3   # ms → s

# Shape of each array: (n_times, n_samples, n_radial) = (1, 1024, 129)
pf_ne   = cdf['n_e']['data']     / 1e20   # m^-3  → 10^20/m^3
pf_te   = cdf['T_e']['data']     / 1e3    # eV    → keV
pf_ti   = cdf['T_12C6']['data']  / 1e3    # eV    → keV
pf_psin = cdf['psi_n']['data']            # (n_times, n_samples, n_radial)

# Decide what form of IDA we are dealing with (this is adjustments to changing formats of IDA-lite, will evenutually be standardized)
if len(np.shape(pf_ne )) == 3:
      print('Assumming single time, posterior distribution')
      workflow_num = 1
elif len(np.shape(pf_ne )) == 2:
      print('Assumming multiple times, with written uncertainties')
      workflow_num = 2
else:
      print('WARNING - SHAPE NOT RECOGNIZED!!')
      workflow_num = 0

# Select matching time index
if workflow_num == 1:
      t_idx      = 0
elif workflow_num == 2:
      t_idx = np.argmin(np.abs(pf_time-sim_time/1e3))
      print(f'Time = time = {pf_time[t_idx]}s, wanted time = {sim_time}ms')

ne_samples = pf_ne[t_idx]    # (1024, 129)
te_samples = pf_te[t_idx]
ti_samples = pf_ti[t_idx]
if workflow_num == 1:
      psin_ida = pf_psin[t_idx, 0, :]   # radial grid is the same for all samples
elif workflow_num == 2:
      psin_ida = pf_psin

print(f"IDA grid: {psin_ida.shape[0]} radial points, "
      f"psiN = [{psin_ida[0]:.4f}, {psin_ida[-1]:.4f}]")
if workflow_num == 1:
      print(f"Posterior samples: {ne_samples.shape[0]}")
      print(f"ne(0) mean  = {ne_samples[:, 0].mean():.4f} x 10^20/m^3")
      print(f"te(0) mean  = {te_samples[:, 0].mean():.4f} keV")
      print(f"ti(0) mean  = {ti_samples[:, 0].mean():.4f} keV")
elif workflow_num == 2:      
      print(f"Will load uncertainties separately later")
      print(f"ne(0) mean  = {ne_samples.mean():.4f} x 10^20/m^3")
      print(f"te(0) mean  = {te_samples.mean():.4f} keV")
      print(f"ti(0) mean  = {ti_samples.mean():.4f} keV")      
      
# raise Exception("THE TIME HERE IS POTENTIALLY A PROBLEM - REREQUEST IDA OR CHECK RAW DATA")

KeyError: 'time'

In [ ]:
def summarise_samples(samples, method=UNCERTAINTY_METHOD):
    """
    Reduce (n_samples, n_radial) posterior draws to a central value and
    a 1σ-equivalent uncertainty band.

    Parameters
    ----------
    samples : ndarray, shape (n_samples, n_radial)
    method  : 'std' or 'percentile'

    Returns
    -------
    centre : ndarray (n_radial,)   – mean or median
    lo     : ndarray (n_radial,)   – centre − 1σ  (or 16th percentile)
    hi     : ndarray (n_radial,)   – centre + 1σ  (or 84th percentile)
    sigma  : ndarray (n_radial,)   – half-width of the band (hi − centre)
    """
    if method == 'std':
        centre = samples.mean(axis=0)
        sigma  = samples.std(axis=0)
        lo     = centre - sigma
        hi     = centre + sigma
    else:  # percentile
        centre = np.median(samples, axis=0)
        lo     = np.percentile(samples, 16, axis=0)
        hi     = np.percentile(samples, 84, axis=0)
        sigma  = (hi - lo) / 2.0   # symmetric half-width for downstream use
    return centre, lo, hi, sigma

In [ ]:
# ---- Summarise posteriors ----
if workflow_num == 1:
      ne_centre, ne_lo, ne_hi, sigma_ne = summarise_samples(ne_samples)
      te_centre, te_lo, te_hi, sigma_te = summarise_samples(te_samples)
      ti_centre, ti_lo, ti_hi, sigma_ti = summarise_samples(ti_samples)
elif workflow_num == 2:
      ne_centre = ne_samples.copy()
      te_centre = te_samples.copy()
      ti_centre = ti_samples.copy()
      
      sigma_ne   = cdf['n_e_err']['data'][t_idx]     / 1e20   # m^-3  → 10^20/m^3
      sigma_te   = cdf['T_e_err']['data'][t_idx]     / 1e3    # eV    → keV
      sigma_ti   = cdf['T_12C6_err']['data'][t_idx]  / 1e3    # eV    → keV
      
      ne_lo = ne_centre - sigma_ne
      te_lo = te_centre - sigma_te
      ti_lo = ti_centre - sigma_ti
            
      ne_hi = ne_centre + sigma_ne
      te_hi = te_centre + sigma_te
      ti_hi = ti_centre + sigma_ti

# ni: derive from ne uncertainty (quasi-neutrality)
# Assumes ni ≈ ne (or scale by Zeff if available)
sigma_ni  = sigma_ne.copy()
ni_centre = ne_centre.copy()

print(f"\nne sigma: core={sigma_ne[0]:.4f}, mid={sigma_ne[len(psin_ida)//2]:.4f}, "
      f"edge={sigma_ne[-5]:.4f}  (10^20/m^3)")
print(f"te sigma: core={sigma_te[0]:.4f}, mid={sigma_te[len(psin_ida)//2]:.4f}, "
      f"edge={sigma_te[-5]:.4f}  (keV)")
print(f"ti sigma: core={sigma_ti[0]:.4f}, mid={sigma_ti[len(psin_ida)//2]:.4f}, "
      f"edge={sigma_ti[-5]:.4f}  (keV)")

mid = len(psin_ida) // 2
print(f"\nFractional uncertainties at psiN≈0.5:")
print(f"  ne: {sigma_ne[mid]/max(ne_centre[mid], 1e-30):.1%}")
print(f"  te: {sigma_te[mid]/max(te_centre[mid], 1e-30):.1%}")
print(f"  ti: {sigma_ti[mid]/max(ti_centre[mid], 1e-30):.1%}")

In [ ]:
# ---- Load g-file to get the target psi_N grid ----
eqdsk = read_geqdsk(geqdsk_file)
psi_N = eqdsk.psi_N  # uniform grid, 0 to 1

print(f"g-file psi_N grid: {len(psi_N)} points")

# ---- Interpolate IDA summaries onto g-file grid (for bouquet) ----
def interp_to_grid(psin_src, arr, psin_tgt):
    return interp1d(psin_src, arr,
                    kind='linear', bounds_error=False,
                    fill_value=(arr[0], arr[-1]))(psin_tgt)

ne_fit_grid    = interp_to_grid(psin_ida, ne_centre, psi_N)
te_fit_grid    = interp_to_grid(psin_ida, te_centre, psi_N)
ti_fit_grid    = interp_to_grid(psin_ida, ti_centre, psi_N)
sigma_ne_grid  = interp_to_grid(psin_ida, sigma_ne,  psi_N)
sigma_te_grid  = interp_to_grid(psin_ida, sigma_te,  psi_N)
sigma_ti_grid  = interp_to_grid(psin_ida, sigma_ti,  psi_N)

### 3a. Diagnostic plots: raw data, fitted profiles, and uncertainty envelopes

These plots show:
- **Left column:** Raw measurement data (dots) overlaid on the fitted p-file
  profile (black line) with the computed ±1σ band (shaded).
- **Right column:** The smooth 1σ uncertainty envelope $\sigma(\psi_N)$.

This verifies that the uncertainty envelopes faithfully capture the
measurement scatter.

In [ ]:
# Thin out samples for the spaghetti overlay (all would be slow)
N_SPAGHETTI = 100
rng = np.random.default_rng(42)
idx = rng.choice(ne_samples.shape[0], size=N_SPAGHETTI, replace=False)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

if workflow_num == 1:
    profiles_info = [
        ('$n_e$', r'($10^{20}$/m$^3$)',
        ne_samples[idx], ne_centre, ne_lo, ne_hi, sigma_ne),
        ('$T_e$', '(keV)',
        te_samples[idx], te_centre, te_lo, te_hi, sigma_te),
        ('$T_i$', '(keV)',
        ti_samples[idx], ti_centre, ti_lo, ti_hi, sigma_ti),
    ]
elif workflow_num == 2:
    profiles_info = [
        ('$n_e$', r'($10^{20}$/m$^3$)',
        [ne_samples], ne_centre, ne_lo, ne_hi, sigma_ne),
        ('$T_e$', '(keV)',
        [te_samples], te_centre, te_lo, te_hi, sigma_te),
        ('$T_i$', '(keV)',
        [ti_samples], ti_centre, ti_lo, ti_hi, sigma_ti),
    ]

band_label = (r'16th–84th pct' if UNCERTAINTY_METHOD == 'percentile'
              else r'mean $\pm 1\sigma$')

for row, (name, unit, spaghetti, centre, lo, hi, sigma) in enumerate(profiles_info):

    # ---- Left: posterior spaghetti + central value + band ----
    ax = axes[row, 0]
    for s in spaghetti:
        ax.plot(psin_ida, s, color='tab:blue', alpha=0.05, lw=0.6)
    ax.fill_between(psin_ida, lo, hi,
                    alpha=0.35, color='tab:orange', label=band_label)
    ax.plot(psin_ida, centre, 'k-', lw=2,
            label='Median' if UNCERTAINTY_METHOD == 'percentile' else 'Mean')
    ax.set_ylabel(f'{name} {unit}')
    ax.set_xlim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if row == 0:
        ax.set_title('Posterior samples + central value + uncertainty band')

    # ---- Right: uncertainty width + fractional uncertainty ----
    ax = axes[row, 1]
    ax.plot(psin_ida, sigma, 'r-', lw=2, label=r'$1\sigma$ half-width')
    ax.set_ylabel(f'$\\sigma$ {unit}', color='r')
    ax.tick_params(axis='y', labelcolor='r')

    ax2 = ax.twinx()
    frac = sigma / np.maximum(np.abs(centre), 1e-30)
    ax2.plot(psin_ida, frac * 100, 'b--', lw=1, alpha=0.7)
    ax2.set_ylabel('Fractional (%)', color='b', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='b')

    ax.set_xlim(0, 1.0)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)
    if row == 0:
        ax.set_title(r'$1\sigma$ envelope from posterior samples')

for ax in axes[-1, :]:
    ax.set_xlabel(r'$\psi_N$')

method_str = 'percentile (16/84)' if UNCERTAINTY_METHOD == 'percentile' else 'mean ± std'
fig.suptitle(f'IDA posterior samples — uncertainty summary ({method_str})',
             fontsize=14, y=1.01)
plt.tight_layout()

fig.savefig(os.path.join(FIG_DIR, 'uncertainty_estimation.png'),
            dpi=150, bbox_inches='tight')
print(f"Saved: {FIG_DIR}/uncertainty_estimation.png")
plt.show()

In [ ]:
plt.plot(sigma_ne, label='ne sigma')

## 4. Initialize TokaMaker

We use the pre-built DIII-D mesh (`DIIID_mesh.h5`), define conductor
and coil regions, and call `setup()` with the toroidal field
$F_0 = R_0 B_0$ read from the g-file.

> **Note:** Only one `TokaMaker` instance can exist per Python kernel.

In [ ]:
myOFT = OFT_env(nthreads=2)
mygs  = TokaMaker(myOFT)

# Load DIII-D mesh
mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh(f'{MESH_FILE}')
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)

# Read F0 = R0*B0 from the g-file
eqdsk_ref = read_geqdsk(geqdsk_file)
F0 = abs(eqdsk_ref.R_center * eqdsk_ref.B_center)

mygs.setup(order=3, F0=F0)
mygs.settings.maxits = 800
mygs.settings.pm = False
mygs.update_settings()

# Vertical stability coil pair (F9A/F9B antisymmetric)
mygs.set_coil_vsc({'F9A': 1.0, 'F9B': -1.0})

print(f'F0 = {F0:.4f} T·m')
print(f'DIII-D mesh loaded: {len(mesh_pts)} points')
print('TokaMaker initialized for DIII-D.')

In [ ]:
# ---- Isoflux from g-file boundary ----
# Use the 199749 g-file boundary shape as isoflux targets
isoflux_pts     = np.column_stack([eqdsk_ref.boundary_R, eqdsk_ref.boundary_Z])
isoflux_weights = np.ones(len(isoflux_pts)) * 500.0
mygs.set_isoflux(isoflux_pts, weights=isoflux_weights)

print(f"Isoflux: {len(isoflux_pts)} target points from g199749.03350 boundary")

In [ ]:
# ---- DIII-D coil regularisation ----
# Use weak regularisation toward zero for all coils.
# The inverse solver in reconstruct_equilibrium will find
# appropriate coil currents to match the g-file boundary.
regularization_terms = []
for name in mygs.coil_sets:
    regularization_terms.append(
        mygs.coil_reg_term({name: 1.0}, target=0.0, weight=1.0))

# Small weight on the virtual VSC to allow up-down adjustment
regularization_terms.append(
    mygs.coil_reg_term({'#VSC': 1.0}, target=0.0, weight=1e-2))

mygs.set_coil_reg(reg_terms=regularization_terms)

print(f'Coil regularisation set for {len(mygs.coil_sets)} DIII-D coil sets.')
print('TokaMaker fully initialized for DIII-D.')

## 5. Reconstruct baseline equilibrium

We reconstruct the baseline equilibrium (shot 199749, 3350 ms) in
TokaMaker using `reconstruct_equilibrium()`.  This:

1. Decomposes $j_\phi$ into bootstrap ($j_{\text{BS}}$) and inductive
   ($j_{\text{ind}}$) components via TokaMaker's Sauter model.
2. Iterates on the inductive scale factor to match the g-file $l_i$.
3. Corrects residual $I_p$ drift.

The kinetic profiles are read from the p-file and converted to SI units
($n_e$ in m$^{-3}$, $T_e$ in eV) as required by the reconstruction routine.

In [ ]:
# ---- Prepare SI profiles from IDA posterior means ----

# --- Load Zeff samples from IDA ---
if workflow_num == 1:
    zeff_samples = cdf['Zeff']['data'][t_idx]          # (n_samples, n_radial)
    zeff_centre  = zeff_samples.mean(axis=0)
    zeff_centre  = np.clip(zeff_centre, 1.0, None)
elif workflow_num == 2:
    zeff_samples = cdf['Zeff']['data'][t_idx]          # (n_samples, n_radial)
    zeff_centre  = np.clip(zeff_samples, 1.0, None)

# --- Interpolate onto g-file psi_N grid in SI units ---
ne_SI   = interp_to_grid(psin_ida, ne_centre,   psi_N) * 1e20   # m^-3
te_SI   = interp_to_grid(psin_ida, te_centre,   psi_N) * 1e3    # eV
ni_SI   = interp_to_grid(psin_ida, ne_centre,   psi_N) * 1e20   # m^-3  (quasi-neutrality approx)
ti_SI   = interp_to_grid(psin_ida, ti_centre,   psi_N) * 1e3    # eV
Zeff_eq = interp_to_grid(psin_ida, zeff_centre, psi_N)
Zeff_eq = np.clip(Zeff_eq, 1.0, None)

print(f"Profiles interpolated onto {len(psi_N)}-point psi_N grid.")
print(f"ne(0) = {ne_SI[0]:.3e} m^-3")
print(f"te(0) = {te_SI[0]:.1f} eV")
print(f"Zeff(0) = {Zeff_eq[0]:.2f}")
print(f"Ip    = {eqdsk.Ip:.3e} A")

# --- Kinetic profiles on IDA grid (including SOL) for bouquet ---
ne_SI_kin   = ne_centre * 1e20     # m^-3
te_SI_kin   = te_centre * 1e3      # eV
ni_SI_kin   = ne_centre * 1e20     # m^-3 (quasi-neutrality approx)
ti_SI_kin   = ti_centre * 1e3      # eV
print(f"IDA kinetic grid: {len(psin_ida)} points, psiN = [{psin_ida[0]:.4f}, {psin_ida[-1]:.4f}]")

In [ ]:
# ---- Set isoflux targets from 199749 boundary shape ----
isoflux_pts_eq = np.column_stack([eqdsk.boundary_R, eqdsk.boundary_Z])
isoflux_weights_eq = np.ones(len(eqdsk.boundary_R)) * 200.0
mygs.set_isoflux(isoflux_pts_eq, weights=isoflux_weights_eq)

# ---- Initial guess for inductive j_phi shape ----
nr = len(psi_N)
guess_jinductive = create_power_flux_fun(nr, 1.5, 1.5)['y']

# ---- Reconstruct ----
print("Starting equilibrium reconstruction...")
print("This may take a few minutes.\n")

result = reconstruct_equilibrium(
    mygs, eqdsk,
    ne_SI, te_SI, ni_SI, ti_SI, Zeff_eq,
    isoflux_pts_eq, isoflux_weights_eq, pad_psi,
    guess_jinductive=guess_jinductive,
    n_k=5,
    psi_bridge=0.99,
    rescale_j_BS=False,
    shelf_psi_N=0.0,
    initialize_psi=True,
)

print("\nReconstruction complete.")

In [ ]:
fix, ax = plt.subplots(1,1)
mygs.plot_machine(fig,ax,coil_colormap='seismic',coil_scale=1.E-6,coil_clabel=r'$I_C$ [MA]',coil_symmap=True)
mygs.plot_constraints(fig, ax)
mygs.plot_psi(fig,ax,plasma_nlevels=10,vacuum_nlevels=10)

In [ ]:
# ---- Store baseline equilibrium in HDF5 database ----
db_path = f"{header}.h5"
if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Deleted existing {db_path}")

db = initialize_equilibrium_database(header)

eqdsk_out = f"{header}_baseline.geqdsk"
eqdsk_out_abs = os.path.abspath(eqdsk_out)
mygs.save_eqdsk(eqdsk_out, nr=257, nz=257,
                truncate_eq=False, lcfs_pad=pad_psi)

li1 = mygs.get_stats(lcfs_pad=pad_psi, li_normalization='std')['l_i']
li3 = mygs.get_stats(lcfs_pad=pad_psi, li_normalization='iter')['l_i']

store_equilibrium(
    header, 0, eqdsk_out_abs,
    psi_N,
    result['j_phi_fit'],
    result['j_BS_used'],
    result['j_inductive_fit'],
    ne_SI, te_SI, ni_SI, ti_SI,
    np.zeros_like(ti_SI),  # w_ExB placeholder
    li1, li3,
)

# os.remove(eqdsk_out)
print(f"Baseline stored: li(1)={li1:.4f}, li(3)={li3:.4f}")

## 6. Verify reconstruction

Compare the TokaMaker reconstruction against the original g-file to
ensure fidelity before generating perturbations.

In [ ]:
all_results = {geqdsk_file: result}
plot_tokamaker_comparison(mygs, all_results, plot_idx=0)

plt.savefig(os.path.join(FIG_DIR, 'reconstruction_comparison.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {FIG_DIR}/reconstruction_comparison.png")
plt.show()

## 6.5  σ→0 sanity preview (coil-drift workflow)

Quick verification that the coil-drift workflow reproduces the
reconstructed baseline equilibrium when fed back its own profiles.
Installs ±1 % hard bounds on every coil (and on the `#VSC` channel),
solves once, and reports axis/Ip/coil-drift residuals.


In [ ]:
import numpy as np

# ---- Capture recon snapshot (for the post-solve diagnostic only) ----
_coils_recon, _ = mygs.get_coil_currents()
recon_coils = {k: float(v) for k, v in _coils_recon.items()}
recon_psi   = mygs.get_psi(False).copy()
recon_axis  = (float(mygs.o_point[0]), float(mygs.o_point[1]))
recon_Ip    = float(mygs.get_globals()[0])

# ---- Build pp_prof / ffp_prof exactly as generate_bouquet's
#       internal q-baseline-check does (TokaMaker_interface.py).
#       psi_N_kinetic=psin_ida is the bouquet convention.
EC = 1.602176634e-19
recon_pressure = EC * (ne_SI * te_SI + ni_SI * ti_SI)

psi_range = mygs.psi_bounds[1] - mygs.psi_bounds[0]
pp_y = np.gradient(recon_pressure) / (np.gradient(psi_N) * psi_range)
pp_y[-1] = 0.0
pp_prof  = {'type': 'linterp',      'y': pp_y,                       'x': psi_N}
ffp_prof = {'type': 'jphi-linterp', 'y': result['j_phi_fit'].copy(), 'x': psi_N}

# ---- Install split-budget hard bounds (matches generate_bouquet) ----
COIL_DRIFT          = 0.01
COIL_DRIFT_FLOOR_A  = 50.0
VSC_COILS           = ('F9A', 'F9B')
VSC_BUDGET_FRAC     = 0.5    # half the budget to bare, half to #VSC

_vsc_in_set = tuple(c for c in VSC_COILS if c in recon_coils)
_alpha = COIL_DRIFT * (1.0 - VSC_BUDGET_FRAC)
_beta  = COIL_DRIFT * VSC_BUDGET_FRAC

bounds = {}
for n, b in recon_coils.items():
    if n in _vsc_in_set:
        delta = max(_alpha * abs(b), COIL_DRIFT_FLOOR_A)
    else:
        delta = max(COIL_DRIFT * abs(b), COIL_DRIFT_FLOOR_A)
    bounds[n] = [b - delta, b + delta]
if len(_vsc_in_set) > 0:
    _vsc_min = min(abs(recon_coils[c]) for c in _vsc_in_set)
    _vsc_delta = max(_beta * _vsc_min, COIL_DRIFT_FLOOR_A)
    bounds['#VSC'] = [-_vsc_delta, _vsc_delta]
mygs.set_coil_bounds(bounds)

# ---- σ→0 forward solve.  Use abs(eqdsk.Ip) for the Ip target,
#       matching generate_bouquet's q-baseline-check convention.
mygs.set_targets(Ip=abs(eqdsk.Ip), pax=float(recon_pressure[0]))
mygs.set_profiles(pp_prof=pp_prof, ffp_prof=ffp_prof)
try:
    mygs.solve()
    s0_ok, s0_err = True, None
except Exception as exc:
    s0_ok, s0_err = False, str(exc)

if s0_ok:
    s0_axis  = (float(mygs.o_point[0]), float(mygs.o_point[1]))
    s0_Ip    = float(mygs.get_globals()[0])
    s0_coils = {k: float(v) for k, v in mygs.get_coil_currents()[0].items()}
    drifts = {n: s0_coils[n] - recon_coils[n] for n in recon_coils}
    daR = (s0_axis[0] - recon_axis[0]) * 1e3
    daZ = (s0_axis[1] - recon_axis[1]) * 1e3
    print(f"σ→0 sanity preview (±{COIL_DRIFT*100:.1f}% total drift, "
          f"vsc_budget_frac={VSC_BUDGET_FRAC:.2f}):")
    print(f"  recon axis = ({recon_axis[0]:.4f}, {recon_axis[1]:+.5f})")
    print(f"  σ→0  axis  = ({s0_axis[0]:.4f}, {s0_axis[1]:+.5f})  Δ=({daR:+.2f}, {daZ:+.2f}) mm")
    print(f"  recon Ip   = {recon_Ip:+.0f}    (eqdsk Ip target = {abs(eqdsk.Ip):+.0f})")
    print(f"  σ→0  Ip    = {s0_Ip:+.0f}  Δ vs recon = {s0_Ip - recon_Ip:+.0f} A "
          f"({100*(s0_Ip - recon_Ip)/abs(recon_Ip):+.3f}%)")
    print(f"  top coil drifts (worst-case total bound is ±{COIL_DRIFT*100:.1f}% of |baseline|):")
    for n, d in sorted(drifts.items(), key=lambda kv: -abs(kv[1]))[:5]:
        pct = 100*d/recon_coils[n] if recon_coils[n] else 0.0
        tag = ' (VSC pair)' if n in _vsc_in_set else ''
        print(f"    {n:6s}: {d:+8.0f} A  ({pct:+.2f}%{tag})")
else:
    print(f"σ→0 sanity preview FAILED: {s0_err}")

# Reset mygs to baseline so generate_bouquet starts from a clean state.
mygs.set_coil_currents(recon_coils)
mygs.set_psi(recon_psi)
mygs.set_coil_bounds(None)   # generate_bouquet will reinstall its own bounds


## 7. Generate perturbed equilibrium family

We now generate **15 perturbed equilibria** using `generate_bouquet()`.

### Uncertainty inputs

- **$n_e$, $T_e$, $T_i$:** Data-driven 1σ envelopes computed in §3 from
  raw measurement scatter. These are converted to SI units (m$^{-3}$, eV)
  to match the profile units used by bouquet.
- **$n_i$:** Derived from $n_e$ uncertainty via quasi-neutrality.
- **$j_\phi$:** Flat 10% fractional envelope (standard).
- **Length scales:** Kept at standard SPARC values ($n$: 0.6, $T$: 0.5,
  $j$: 0.3 in $\psi_N$ units).

Each perturbed equilibrium:
1. Draws GPR-perturbed profiles within the uncertainty envelopes.
2. Recomputes bootstrap current.
3. Iterates to match baseline $I_p$ and $l_i$.

In [ ]:
# ---- Convert posterior uncertainties to SI on g-file grid ----
# sigma_*_grid are in 10^20/m^3 (ne/ni) and keV (te/ti), on psi_N grid

sigma_ne_SI = sigma_ne_grid * 1e20   # m^-3
sigma_te_SI = sigma_te_grid * 1e3    # eV
sigma_ni_SI = sigma_ne_grid * 1e20   # m^-3  (same fractional uncertainty as ne)
sigma_ti_SI = sigma_ti_grid * 1e3    # eV

# --- Uncertainties on IDA grid (including SOL) ---
sigma_ne_SI_kin = sigma_ne * 1e20   # m^-3
sigma_te_SI_kin = sigma_te * 1e3    # eV
sigma_ni_SI_kin = sigma_ne * 1e20   # m^-3
sigma_ti_SI_kin = sigma_ti * 1e3    # eV

# j_phi: flat fractional envelope (unchanged)
sigma_jphi = frac_jphi * np.abs(result['j_phi_fit'])

print("Uncertainty profiles converted to SI:")
print(f"  sigma_ne(0): {sigma_ne_SI[0]:.3e} m^-3  ({sigma_ne_SI[0]/ne_SI[0]*100:.1f}% of ne)")
print(f"  sigma_te(0): {sigma_te_SI[0]:.1f} eV  ({sigma_te_SI[0]/te_SI[0]*100:.1f}% of te)")
print(f"  sigma_ti(0): {sigma_ti_SI[0]:.1f} eV  ({sigma_ti_SI[0]/ti_SI[0]*100:.1f}% of ti)")
print(f"  sigma_jphi:  flat {frac_jphi*100:.0f}%")

In [ ]:
def ida_to_pfile(psin, ne, te, ti, nc, zeff,
                 omega_tor=None,
                 ion_N=ion_N, ion_Z=ion_Z, ion_A=ion_A):
    """
    Build a PFile object from IDA posterior-mean profiles.

    Parameters
    ----------
    psin  : (n,) array  – psiN grid
    ne    : (n,) array  – electron density     [10^20/m^3]
    te    : (n,) array  – electron temperature [keV]
    ti    : (n,) array  – ion temperature      [keV]
    nc    : (n,) array  – carbon density       [10^20/m^3]
    zeff  : (n,) array  – effective charge     [dimensionless]
    omega_tor : (n,) array or None
                        – toroidal rotation    [kRad/s]
                          (omega_tor_12C6 from IDA, converted)

    Returns
    -------
    PFile object with ne, te, ni, ti, nz1, ptot and ion species set.
    """

    # ni from quasi-neutrality: ne = ni + Z_C * n_C
    Z_C = ion_Z[0]   # = 6 for carbon
    ni  = np.maximum(ne - Z_C * nc, 0.0)

    pf = PFile.new()
    pf.set_ion_species(N=ion_N, Z=ion_Z, A=ion_A)

    pf.set_profile('ne',  psin, ne)
    pf.set_profile('te',  psin, te)
    pf.set_profile('ni',  psin, ni)
    pf.set_profile('ti',  psin, ti)
    pf.set_profile('nz1', psin, nc)   # carbon impurity density

    # Pressure (needed by some bouquet routines)
    pf.compute_pressure()

    # Toroidal rotation (IDA omega_tor_12C6 is already in rad/s — convert)
    if omega_tor is not None:
        pf.set_profile('omeg', psin, omega_tor / 1e3,   # rad/s → kRad/s
                       units='kRad/s')

    return pf

# ---- Build PFile from IDA posterior means ----
nc_samples  = cdf['n_12C6']['data'][t_idx]        # (n_samples, n_radial) m^-3
if workflow_num == 1:
    nc_centre   = nc_samples.mean(axis=0) / 1e20      # → 10^20/m^3
elif workflow_num == 2:
    nc_centre   = nc_samples.copy() / 1e20            # → 10^20/m^3

omega_samples = cdf['omega_tor_12C6']['data'][t_idx]  # rad/s
if workflow_num == 1:
    omega_centre  = omega_samples.mean(axis=0) / 1e3  # → kRad/s
elif workflow_num == 2:
    omega_centre  = omega_samples.copy() / 1e3        # → kRad/s

ida_pfile = ida_to_pfile(
    psin_ida, ne_centre, te_centre, ti_centre,
    nc_centre, zeff_centre,
    omega_tor=omega_centre * 1e3,   # pass in rad/s; function divides by 1e3
)

# Serialise to bytes — drop-in replacement for pfile_raw_bytes everywhere
ida_pfile_bytes = ida_pfile.to_bytes()

print("PFile built from IDA data:")
print(f"  ne(0) = {ida_pfile.ne[0]:.4f} x 10^20/m^3")
print(f"  te(0) = {ida_pfile.te[0]:.4f} keV")
print(f"  ti(0) = {ida_pfile.ti[0]:.4f} keV")
print(f"  ni(0) = {ida_pfile.ni[0]:.4f} x 10^20/m^3")
print(f"  omeg(0) = {ida_pfile.omeg[0]:.4f} x kRad/s")


In [ ]:
# ---- Targets from reconstruction ----
Ip_target  = abs(eqdsk.Ip)
l_i_target = mygs.get_stats(lcfs_pad=pad_psi, li_normalization='std')['l_i']

# Read baseline files as raw bytes for HDF5 archival
with open(geqdsk_file, 'rb') as fh:
    baseline_eqdsk_raw = fh.read()
with open(IDA_file, 'rb') as fh:
    ida_raw_bytes = fh.read()

# Restore isoflux targets
mygs.set_isoflux(result['isoflux_pts'], weights=result['weights'])

print(f"Generating {n_equils} perturbed equilibria...")
print(f"  Ip target:  {Ip_target/1e6:.3f} MA")
print(f"  li target:  {l_i_target:.4f}")
print(f"  jBS scale:  {jBS_scale_range}")
print()

# Homotopy hard-bound flow (Pass 1 loose -> Pass 3 strict, warm-started):
#   coil_drift=0.01 is the baseline drift used when homotopy_passes
#   is None (legacy single-pass).  With homotopy_passes provided,
#   each tuple (drift_F_bare, drift_VSC_channel) defines a pass and
#   the QP progressively tightens.  Draws that fail tighter passes
#   are rolled back to the last successful pass and recorded with
#   in_spec=False (so out-of-spec draws are still archived for
#   downstream analysis; filter on `in_spec` attr to use only the
#   tight-spec subset).  Set homotopy_passes=None for legacy.
diagnostics = generate_bouquet(
    mygs, psi_N, n_equils, header,
    result['j_phi_fit'],
    ne_SI_kin, te_SI_kin, ni_SI_kin, ti_SI_kin,
    sigma_ne_SI_kin, sigma_te_SI_kin, sigma_ni_SI_kin, sigma_ti_SI_kin, sigma_jphi,
    n_ls, t_ls, j_ls,
    Ip_target, l_i_target, Zeff_eq,
    input_jinductive=result['j_inductive_fit'],
    l_i_tolerance=10,
    l_i_proxy_threshold=12.5,
    psi_pad=pad_psi,
    constrain_sawteeth=False,
    recalculate_j_BS=True,
    jBS_scale_range=jBS_scale_range,
    pfile_bytes=ida_pfile_bytes,
    baseline_eqdsk_bytes=baseline_eqdsk_raw,
    baseline_pfile_bytes=ida_pfile_bytes,
    diagnostic_plots=False,
    scan_val=0,
    psi_N_kinetic=psin_ida,
    coil_drift=0.01,
    # ---- Progressive hard-bound homotopy ----
    # Per-pass (drift_F_bare, drift_VSC_channel) tuples.  Total F9
    # drift = bare + VSC; non-VSC F-coils total = bare alone.
    #   Pass 1 (0.05, 0.10): loose start so QP can find feasible region
    #   Pass 2 (0.02, 0.05): intermediate tightening
    #   Pass 3 (0.01, 0.01): strict global -- max total F9 drift <= 2%,
    #                       non-VSC F-coils <= 1%
    # Each pass warm-starts psi from prior pass's converged solution.
    # On failure rolls back to last successful pass (recorded per
    # draw via H5 attrs homotopy_pass, max_F_drift_pct, etc).
    homotopy_passes=[(0.05, 0.10), (0.02, 0.05), (0.01, 0.01)],
    # ---- In-spec criterion (strict +/- 2% global) ----
    inspec_F_max=0.02,
    inspec_VSC_max=0.02,
    # ---- Pressure-match tolerance (relaxed to match DIII-D <P>
    # measurement uncertainty: diamagnetic ~10%, kinetic EFIT ~5%).
    # 0.5% (old default) was 10-20x tighter than measurement supports
    # and over-constrained kinetic sampling.
    p_thresh=5.0,
    # ---- Soft regularization weight on the #VSC channel ----
    # Pulls F9A/F9B antisymmetric channel toward zero in addition to
    # the per-coil bare soft-reg (weight=1e4) that pulls F9A and F9B
    # individually toward recon.  At sigma~1 the plasma centroid moves
    # and F9 must track -- use VSC_W=1.0.  At sigma~0.5 a moderate
    # antisymmetric pull (VSC_W=100) tightens in-spec yield by ~17pp
    # without breaking the QP.
    # WARNING: at sigma=0 with hard bounds + iso-update active, F9
    # naturally drifts ~3% to satisfy the slightly-shifted boundary --
    # no value of VSC_W rescues this (VSC_W>=1e3 breaks the QP, smaller
    # values don't help).  For sigma=0 strict in-spec, set
    # homotopy_passes=None (soft-reg-only mode); natural F9 drift then
    # is ~0.6%.  Empirical guideline based on DIII-D 204441@4400:
    #     sigma=0.0  ->  homotopy_passes=None, VSC_W=1.0
    #     sigma=0.5  ->  homotopy_passes=[...],  VSC_W=100
    #     sigma=1.0  ->  homotopy_passes=[...],  VSC_W=1.0
    vsc_soft_reg_weight=1.0,
)

print(f"\n{len(diagnostics)} perturbed equilibria generated and archived.")


## 8. Visualize the bouquet

Plot the family of perturbed profiles (kinetic, pressure, and $j_\phi$)
overlaid on the baseline with uncertainty bands.

In [ ]:
# Plot all bouquet panels
plot_bouquet(header, scan_value=0, mode='all')
plt.savefig(os.path.join(FIG_DIR, 'bouquet_all_profiles.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {FIG_DIR}/bouquet_all_profiles.png")
plt.show()

In [ ]:
figs = plot_traces(header, scan_value='all')

## 8a. Locked-coil diagnostics

With `lock_coils=True`, every perturbed equilibrium is solved as a
forward Grad-Shafranov problem at fixed coil currents (no isoflux,
no saddle, no inverse current adjustment).  This isolates the effect
of profile perturbations on the plasma boundary and X-point geometry.

The HDF5 stores three diagnostics per equilibrium:

  * `boundary_devs [mm]` — per-LCFS-point spatial deviation from the
    TokaMaker baseline LCFS (psi-evaluation method, see
    `psi_boundary_deviation`).
  * `x_point_devs [mm]`  — per-baseline-X-point drift; `NaN` entries
    mark baseline X-points whose perturbed counterpart drifted out of
    pairing range or vanished.
  * `coil_drift [A]`     — per-coil ``I_perturbed - I_baseline``.  With
    a hard pin this should be ~0 unless something (e.g. VSC reg) is
    fighting the lock — useful sanity signal during testing.


In [ ]:
# Per-draw diagnostics from the new H5 schema:
#   * boundary RMS/max via point-to-segment distance vs baseline eqdsk
#   * homotopy pass reached (0=Pass1 loose, 1=Pass2, 2=Pass3 strict)
#   * in_spec flag (max F-coil drift <= inspec_F_max AND VSC <= inspec_VSC_max)
#   * max_F_drift_pct, max_VSC_drift_pct (per-coil drift on total current)

import h5py, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/Users/danielburgess/Desktop/plasma/bouquet_coil_bounds')
from bouquet import GEQDSKEquilibrium

def _p2s(p, a, b):
    ap = p - a; ab = b - a
    t = np.clip(np.dot(ap, ab) / max(np.dot(ab, ab), 1e-30), 0, 1)
    return np.linalg.norm(p - (a + t * ab))

def _p2c(p, c):
    n = len(c); return min(_p2s(p, c[i], c[(i+1)%n]) for i in range(n))

def _curve_metric_mm(c1, c2):
    d1 = np.array([_p2c(p, c2) for p in c1])
    d2 = np.array([_p2c(p, c1) for p in c2])
    rms = 0.5*(np.sqrt(np.mean(d1**2)) + np.sqrt(np.mean(d2**2))) * 1e3
    mx  = float(np.max(np.concatenate([d1, d2]))) * 1e3
    return rms, mx

h5file = f"{header}.h5"
SCAN_KEY = '0'

with h5py.File(h5file, 'r') as hf:
    parent = hf[f'scan/{SCAN_KEY}']
    bl = parent['_baseline']
    bl_eq = GEQDSKEquilibrium.from_bytes(
        bytes(np.array(bl['baseline.eqdsk']).tobytes()))
    ref_xy = np.column_stack([bl_eq.boundary_R, bl_eq.boundary_Z])
    inspec_F   = float(bl.attrs.get('inspec_F_max',   0.02))
    inspec_VSC = float(bl.attrs.get('inspec_VSC_max', 0.02))

    counts = sorted(int(k) for k in parent.keys() if k.isdigit())
    bnd_rms = np.full(len(counts), np.nan)
    bnd_max = np.full(len(counts), np.nan)
    pass_idx = np.full(len(counts), -1, dtype=int)
    max_F   = np.full(len(counts), np.nan)
    max_VSC = np.full(len(counts), np.nan)
    in_spec = np.zeros(len(counts), dtype=bool)

    for i, c in enumerate(counts):
        g = parent[str(c)]
        ek = next((kk for kk in g.keys() if kk.endswith('.eqdsk')), None)
        if ek is not None:
            eq = GEQDSKEquilibrium.from_bytes(
                bytes(np.array(g[ek]).tobytes()))
            pxy = np.column_stack([eq.boundary_R, eq.boundary_Z])
            r, m = _curve_metric_mm(ref_xy, pxy)
            bnd_rms[i], bnd_max[i] = r, m
        pass_idx[i] = int(g.attrs.get('homotopy_pass', -1))
        max_F[i]    = float(g.attrs.get('max_F_drift_pct', np.nan))
        max_VSC[i]  = float(g.attrs.get('max_VSC_drift_pct', np.nan))
        in_spec[i]  = bool(g.attrs.get('in_spec', False))

print(f"Equilibria stored:   {len(counts)}")
print(f"In-spec criterion:   max F-drift <= {inspec_F*100:.1f}%, "
      f"max VSC <= {inspec_VSC*100:.1f}%")
print(f"In-spec count:       {in_spec.sum()}/{len(counts)} "
      f"({100*in_spec.sum()/max(len(counts),1):.0f}%)")
pass_labels = ['Pass 1 (loose)', 'Pass 2 (mid)', 'Pass 3 (strict)']
for p in range(3):
    n_p = int(np.sum(pass_idx == p))
    if n_p:
        print(f"  reached {pass_labels[p]:<16s}: {n_p}/{len(counts)}")

x = np.arange(len(counts))
colors = ['tab:green' if s else 'tab:red' for s in in_spec]

fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# Panel 1: boundary RMS/max
w = 0.4
axs[0].bar(x - w/2, bnd_rms, w, color=colors, edgecolor='k', alpha=0.85, label='RMS')
axs[0].bar(x + w/2, bnd_max, w, color=colors, edgecolor='k', alpha=0.4,  label='Max')
axs[0].axhline(2.84, ls='--', color='gray', lw=1.0,
               label="recon's own residual vs eqdsk (2.84 mm)")
axs[0].set_ylabel('Boundary deviation [mm]')
axs[0].set_title('Per-draw diagnostics (green = in_spec, red = out_of_spec)')
axs[0].legend(loc='best', fontsize=8); axs[0].grid(ls=':', alpha=0.4, axis='y')

# Panel 2: max F-coil drift
axs[1].bar(x, max_F, color=colors, edgecolor='k', alpha=0.85)
axs[1].axhline(inspec_F*100, ls='--', color='k', lw=1.2,
               label=f'spec ({inspec_F*100:.1f}%)')
axs[1].set_ylabel('max non-VSC F-coil drift [%]')
axs[1].legend(); axs[1].grid(ls=':', alpha=0.4, axis='y')

# Panel 3: max VSC drift
axs[2].bar(x, max_VSC, color=colors, edgecolor='k', alpha=0.85)
axs[2].axhline(inspec_VSC*100, ls='--', color='k', lw=1.2,
               label=f'spec ({inspec_VSC*100:.1f}%)')
axs[2].set_ylabel('max VSC (F9A/F9B) drift [%]')
axs[2].set_xlabel('Perturbed equilibrium index')
axs[2].legend(); axs[2].grid(ls=':', alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'perturbed_eq_diagnostics.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 8b. Coil current drift and % deviation per equilibrium

Every coil gets a unique (color, marker) combination — colors cycle
through a rainbow colormap and markers cycle through a 12-symbol
palette so adjacent-hue coils are still distinguishable.  The same
style is used in both panels, so each coil can be tracked vertically
between the absolute-Amps and % views.

  * **Top:** drift in absolute Amps; the right unit for setting
    `coil_drift_threshold_A`.  When the threshold is set on the run,
    a `±cap` line is drawn here.
  * **Bottom:** % deviation from baseline; naturally collapses to
    `NaN` for any coil with `|baseline| < 1 A` to avoid div-by-near-
    zero blow-ups.

Adapt `COIL_NAMES` for other tokamaks; ordering also drives the
colour assignment along the rainbow.  A per-coil summary table
(sorted by max |drift|) is printed below the plot for quick
identification of the worst offenders.


In [ ]:
# Per-coil drift plot, separated into three classes:
#   * Non-VSC F-coils (F1-F8 A/B): rainbow colors, spec line at +/-inspec_F_max
#   * VSC pair (F9A/F9B):          black/grey, spec line at +/-inspec_VSC_max
#   * E-coils (ECOILA, ECOILB):    dashed lines, NOT in in_spec criterion
#                                  (bounded by coil_drift_floor_A absolute, so
#                                  their % drift can exceed 2% on small baselines
#                                  -- e.g. ECOILB with baseline ~500A and 50A
#                                  floor gives ~10% relative bound)
#
# Drift = (I_draw - I_baseline) / |I_baseline| * 100, NaN if |I_baseline| < 1A.

import h5py, json as _json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

NON_VSC_F = ['F1A', 'F2A', 'F3A', 'F4A', 'F5A', 'F6A', 'F7A', 'F8A',
             'F1B', 'F2B', 'F3B', 'F4B', 'F5B', 'F6B', 'F7B', 'F8B']
VSC_PAIR  = ['F9A', 'F9B']
E_COILS   = ['ECOILA', 'ECOILB']
ALL_COILS = NON_VSC_F + VSC_PAIR + E_COILS

MARKERS = ['o', 's', 'D', '^', 'v', '<', '>', 'p']
CMAP = plt.get_cmap('gist_rainbow')
F_style = {n: {'color': CMAP(i / max(len(NON_VSC_F)-1, 1)),
               'marker': MARKERS[i % len(MARKERS)]}
           for i, n in enumerate(NON_VSC_F)}
VSC_style = {'F9A': {'color': 'black', 'marker': '*'},
             'F9B': {'color': 'dimgray', 'marker': '*'}}
E_style = {'ECOILA': {'color': 'tab:purple', 'marker': 'X', 'ls': '--'},
           'ECOILB': {'color': 'tab:olive', 'marker': 'X', 'ls': '--'}}

h5file = f"{header}.h5"
records = []
inspec_F = inspec_VSC = None
with h5py.File(h5file, 'r') as hf:
    scan_keys = sorted(hf['scan'].keys()) if 'scan' in hf else []
    for sk in scan_keys:
        parent = hf[f'scan/{sk}']
        bl = parent['_baseline']
        if 'coil_currents [A]' not in bl or 'coil_names' not in bl:
            print(f"  scan={sk}: no baseline coil_currents stored, skipping")
            continue
        ref_names = [s.decode() if isinstance(s, bytes) else s
                     for s in np.array(bl['coil_names'])]
        ref_vals = np.array(bl['coil_currents [A]'])
        ref_dict = dict(zip(ref_names, ref_vals))
        if inspec_F is None:
            inspec_F = float(bl.attrs.get('inspec_F_max', 0.02))
            inspec_VSC = float(bl.attrs.get('inspec_VSC_max', 0.02))
        for c in sorted(int(k) for k in parent.keys() if k.isdigit()):
            g = parent[str(c)]
            if 'coil_currents [A]' not in g:
                continue
            d_names = _json.loads(g.attrs.get('coil_names', '[]'))
            d_vals = np.array(g['coil_currents [A]'])
            d_dict = dict(zip(d_names, d_vals))
            records.append((sk, c, ref_dict, d_dict,
                             bool(g.attrs.get('in_spec', False))))

n_eq = len(records)
print(f"Loaded {n_eq} equilibria across {len(scan_keys)} scan(s).")
print(f"In-spec criterion: max non-VSC F-coil <= {inspec_F*100:.1f}%, "
      f"max VSC <= {inspec_VSC*100:.1f}%")
print(f"NOTE: E-coils are bounded by coil_drift_floor_A absolute (default 50A);")
print(f"      on small-baseline E-coils this can be ~5-10% in % terms -- NOT")
print(f"      a spec violation, just a floor reading.")

def drift_pct(d, ref, coil):
    r = ref.get(coil, np.nan); v = d.get(coil, np.nan)
    if np.isfinite(r) and np.isfinite(v) and abs(r) > 1.0:
        return 100.0 * (v - r) / abs(r)
    return np.nan

if n_eq == 0:
    print("No data to plot.")
else:
    fig, (ax_F, ax_VSC, ax_E) = plt.subplots(3, 1, figsize=(13, 10),
                                              sharex=True)
    x = np.arange(n_eq)
    in_spec_arr = np.array([r[4] for r in records])

    # --- Panel 1: non-VSC F-coils ---
    legend_F = []
    for coil in NON_VSC_F:
        st = F_style[coil]
        ys = [drift_pct(dr, ref, coil) for (_, _, ref, dr, _) in records]
        ax_F.plot(x, ys, '-', color=st['color'], marker=st['marker'],
                   markersize=5, alpha=0.85, lw=1.0)
        legend_F.append(Line2D([0], [0], color=st['color'], marker=st['marker'],
                                markersize=6, lw=1.5, label=coil))
    ax_F.axhline(+inspec_F*100, color='k', ls='--', lw=1.2,
                  label=f'+/-{inspec_F*100:.1f}% spec')
    ax_F.axhline(-inspec_F*100, color='k', ls='--', lw=1.2)
    ax_F.axhline(0, color='k', lw=0.8)
    ax_F.set_ylabel('Non-VSC F-coil drift [%]')
    ax_F.set_title(f'Per-coil drift, {n_eq} equilibria '
                    f'(green band = in_spec, red = out_of_spec)')
    ax_F.grid(ls=':', alpha=0.4, axis='y')

    # --- Panel 2: VSC pair ---
    legend_VSC = []
    for coil in VSC_PAIR:
        st = VSC_style[coil]
        ys = [drift_pct(dr, ref, coil) for (_, _, ref, dr, _) in records]
        ax_VSC.plot(x, ys, '-', color=st['color'], marker=st['marker'],
                     markersize=7, alpha=0.85, lw=1.2)
        legend_VSC.append(Line2D([0], [0], color=st['color'],
                                  marker=st['marker'], markersize=8,
                                  lw=1.5, label=coil))
    ax_VSC.axhline(+inspec_VSC*100, color='k', ls='--', lw=1.2,
                    label=f'+/-{inspec_VSC*100:.1f}% spec')
    ax_VSC.axhline(-inspec_VSC*100, color='k', ls='--', lw=1.2)
    ax_VSC.axhline(0, color='k', lw=0.8)
    ax_VSC.set_ylabel('VSC pair drift [%]')
    ax_VSC.grid(ls=':', alpha=0.4, axis='y')

    # --- Panel 3: E-coils (out of in_spec criterion) ---
    legend_E = []
    for coil in E_COILS:
        st = E_style[coil]
        ys = [drift_pct(dr, ref, coil) for (_, _, ref, dr, _) in records]
        ax_E.plot(x, ys, ls=st['ls'], color=st['color'], marker=st['marker'],
                   markersize=6, alpha=0.85, lw=1.2)
        legend_E.append(Line2D([0], [0], color=st['color'], ls=st['ls'],
                                marker=st['marker'], markersize=7, lw=1.5,
                                label=coil))
    ax_E.axhline(0, color='k', lw=0.8)
    ax_E.set_ylabel('E-coil drift [%]')
    ax_E.set_xlabel('Perturbed equilibrium index')
    ax_E.grid(ls=':', alpha=0.4, axis='y')

    # In-spec shading on all three panels
    for ax in [ax_F, ax_VSC, ax_E]:
        for i, ok in enumerate(in_spec_arr):
            ax.axvspan(i - 0.4, i + 0.4,
                        color=('tab:green' if ok else 'tab:red'),
                        alpha=0.10 if ok else 0.05)

    # Legends
    ax_F.legend(handles=legend_F, loc='center left',
                 bbox_to_anchor=(1.01, 0.5), fontsize=7, ncol=1,
                 title='F1-F8 A/B')
    ax_VSC.legend(handles=legend_VSC, loc='center left',
                   bbox_to_anchor=(1.01, 0.5), fontsize=8, title='VSC')
    ax_E.legend(handles=legend_E, loc='center left',
                 bbox_to_anchor=(1.01, 0.5), fontsize=8,
                 title='E-coils (50A floor)')

    plt.tight_layout(rect=[0, 0, 0.90, 1])
    plt.savefig(os.path.join(FIG_DIR, 'coil_drift_per_eq.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Summary by class
    def _drift_stats(coils):
        deltas_pct = []
        for coil in coils:
            for (_, _, ref, dr, _) in records:
                v = drift_pct(dr, ref, coil)
                if np.isfinite(v):
                    deltas_pct.append(abs(v))
        if not deltas_pct:
            return None
        return (np.median(deltas_pct), np.percentile(deltas_pct, 95),
                max(deltas_pct))

    print()
    print(f'Drift summary by coil class (|drift| %):')
    print(f'  {"class":<22s}  {"median":>7s}  {"95th":>7s}  {"max":>7s}')
    for label, coils in [('Non-VSC F-coils', NON_VSC_F),
                          ('VSC pair (F9A/F9B)', VSC_PAIR),
                          ('E-coils', E_COILS)]:
        s = _drift_stats(coils)
        if s:
            print(f'  {label:<22s}  {s[0]:>6.2f}%  {s[1]:>6.2f}%  {s[2]:>6.2f}%')


In [ ]:
# Individual mode plots for clearer detail
for mode in ['kinetic', 'pressure', 'j-phi']:
    plot_bouquet(header, scan_value=0, mode=mode)
    fname = f'bouquet_{mode.replace("-", "")}.png'
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches='tight')
    print(f"Saved: {FIG_DIR}/{fname}")
    plt.show()

In [ ]:
h5file = f"{header}.h5"

plot_pfile_bouquet(h5path=h5file, scan_val=0, x_coord="psi_N")#count=3, 


## 9. Extract q-profiles and locate q = 4 surface

For each perturbed equilibrium stored in the HDF5 database, we:

1. Load the archived g-file bytes.
2. Read the q-profile from the geqdsk data.
3. Interpolate to find where q = 4 (if the surface exists within
   the plasma).

We also extract the baseline q-profile for comparison.


In [ ]:
def find_q_surface(psi_N_q, q_profile, q_target=4.0):
    """
    Find the psi_N location(s) where q = q_target.

    Uses linear interpolation between grid points where q crosses
    the target value.  Returns all crossings (there may be more
    than one if the q-profile is non-monotonic).

    Parameters
    ----------
    psi_N_q : ndarray
        Normalised flux grid for the q-profile.
    q_profile : ndarray
        Safety factor profile.
    q_target : float
        Target q value to locate.

    Returns
    -------
    list of float
        psi_N location(s) of the q = q_target surface.
        Empty list if the surface is not found.
    """
    crossings = []
    for i in range(len(q_profile) - 1):
        q0, q1 = q_profile[i], q_profile[i + 1]
        # Check if q_target is between q0 and q1
        if (q0 - q_target) * (q1 - q_target) <= 0 and q0 != q1:
            # Linear interpolation
            frac = (q_target - q0) / (q1 - q0)
            psi_crossing = psi_N_q[i] + frac * (psi_N_q[i + 1] - psi_N_q[i])
            crossings.append(psi_crossing)
    return crossings


print("q-surface finding function defined.")

In [ ]:
# ---- Extract baseline q-profile ----
# The baseline g-file q-profile
psi_N_baseline = eqdsk.psi_N
q_baseline = eqdsk.qpsi

q4_baseline = find_q_surface(psi_N_baseline, q_baseline, q_target=4.0)
print(f"Baseline q-profile: q(0) = {q_baseline[0]:.3f}, q(1) = {q_baseline[-1]:.3f}")
if q4_baseline:
    print(f"Baseline q=4 surface at psi_N = {q4_baseline[0]:.4f}")
else:
    print("WARNING: q=4 surface not found in baseline!")
    print(f"  q_max = {np.max(q_baseline):.3f} at psi_N = {psi_N_baseline[np.argmax(q_baseline)]:.4f}")

In [ ]:
# ---- Extract q-profiles from all perturbed equilibria ----
q4_locations = []       # psi_N of q=4 for each perturbed equilibrium
q_profiles_all = []     # all q-profiles for plotting
psiN_profiles_all = []  # corresponding psi_N grids

with h5py.File(f"{header}.h5", 'r') as hf:
    # Dynamically discover HDF5 layout
    # Layout is either: scan/{scan_val}/{count} or flat {count}
    if 'scan' in hf:
        scan_grp = hf['scan']
        scan_keys = list(scan_grp.keys())
        print(f"HDF5 layout: scan/ with keys {scan_keys}")
        equil_grp = scan_grp[scan_keys[0]]
    else:
        equil_grp = hf
        print("HDF5 layout: flat (no scan/ group)")

    eq_keys = sorted([k for k in equil_grp.keys() if k.isdigit()],
                     key=lambda k: int(k))

    print(f"Found {len(eq_keys)} equilibria in database (including baseline)")
    print()

    for eq_key in eq_keys:
        eq_grp = equil_grp[eq_key]
        eq_idx = int(eq_key)

        # Find eqdsk dataset by name ending in '.eqdsk'
        eqdsk_ds = [k for k in eq_grp.keys() if k.endswith('.eqdsk')]
        if eqdsk_ds:
            eqdsk_bytes = bytes(eq_grp[eqdsk_ds[0]][()])
            eq_data = read_eqdsk_from_bytes(eqdsk_bytes, read_geqdsk)

            psi_N_eq = eq_data.psi_N
            q_eq = eq_data.qpsi

            q_profiles_all.append(q_eq)
            psiN_profiles_all.append(psi_N_eq)

            # Find q=4
            q4_locs = find_q_surface(psi_N_eq, q_eq, q_target=4.0)

            label = 'baseline' if eq_idx == 0 else f'perturbed {eq_idx}'
            if q4_locs:
                # Take the outermost crossing (largest psi_N)
                q4_psiN = max(q4_locs)
                q4_locations.append(q4_psiN)
                print(f"  [{eq_key}] ({label}): q=4 at psi_N = {q4_psiN:.4f}")
            else:
                print(f"  [{eq_key}] ({label}): q=4 NOT FOUND (q_max = {np.max(q_eq):.3f})")
        else:
            print(f"  [{eq_key}]: no eqdsk dataset found")

q4_locations = np.array(q4_locations)
print(f"\nq=4 found in {len(q4_locations)} / {len(eq_keys)} equilibria")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot perturbed q-profiles (light blue spaghetti)
for i, (psiN_q, q_prof) in enumerate(zip(psiN_profiles_all, q_profiles_all)):
    if i == 0:
        continue  # skip baseline (plotted separately)
    ax.plot(psiN_q, q_prof, '-', color='tab:blue', alpha=0.3, lw=0.8,
            label='Perturbed' if i == 1 else None)

# Plot baseline q-profile (black, thick)
if len(psiN_profiles_all) > 0:
    ax.plot(psiN_profiles_all[0], q_profiles_all[0], 'k-', lw=2.5,
            label='Baseline', zorder=10)

# q=4 reference line
ax.axhline(y=4.0, color='red', ls='--', lw=1.5, label='q = 4')

# Mark q=4 crossings
for q4_psi in q4_locations:
    ax.plot(q4_psi, 4.0, 'rv', ms=6, alpha=0.6, zorder=5)

ax.set_xlabel(r'$\psi_N$', fontsize=13)
ax.set_ylabel('q (safety factor)', fontsize=13)
ax.set_title('q-profiles: baseline and perturbed equilibria', fontsize=14)
ax.set_xlim(0, 1.0)
ax.set_ylim(0, max(8, np.max([np.max(q) for q in q_profiles_all]) * 1.05))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'q_profiles_comparison.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {FIG_DIR}/q_profiles_comparison.png")
plt.show()

## 11. Histogram of q = 4 surface location and 95% confidence interval

### Statistical approach

With 15 samples, we report:
- **Mean** and **standard deviation** of the q = 4 location.
- **95% confidence interval** computed as the 2.5th and 97.5th
  percentiles of the sample distribution.  For 15 samples this is
  equivalent to approximately the min and max of the sample.

In [ ]:
if len(q4_locations) >= 2:
    # ---- Statistics ----
    q4_mean = np.mean(q4_locations)
    q4_std  = np.std(q4_locations, ddof=1)  # sample std
    q4_lo   = np.percentile(q4_locations, 2.5)
    q4_hi   = np.percentile(q4_locations, 97.5)
    q4_median = np.median(q4_locations)

    print("="*60)
    print("q = 4 surface location statistics")
    print("="*60)
    print(f"  N samples:         {len(q4_locations)}")
    print(f"  Mean:              psi_N = {q4_mean:.4f}")
    print(f"  Median:            psi_N = {q4_median:.4f}")
    print(f"  Std deviation:     {q4_std:.4f}")
    print(f"  95% CI:            [{q4_lo:.4f}, {q4_hi:.4f}]")
    print(f"  Range:             [{np.min(q4_locations):.4f}, {np.max(q4_locations):.4f}]")
    print("="*60)

    # ---- Histogram ----
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    # Choose bins: use ~8-12 bins for 15 samples, or Freedman-Diaconis
    n_bins = min(12, max(5, int(np.sqrt(len(q4_locations)))))
    # Ensure bins span the full range with some padding
    bin_pad = 0.5 * q4_std if q4_std > 0 else 0.01
    bins = np.linspace(q4_mean - 3*bin_pad, q4_mean + 3*bin_pad, n_bins + 1)

    ax.hist(q4_locations, bins=bins, color='steelblue', edgecolor='white',
            alpha=0.8, label='Perturbed equilibria')

    # Mark baseline
    if q4_baseline:
        ax.axvline(q4_baseline[0], color='black', lw=2.5, ls='-',
                   label=f'Baseline: $\\psi_N$ = {q4_baseline[0]:.4f}')

    # Mark mean
    ax.axvline(q4_mean, color='red', lw=2, ls='--',
               label=f'Mean: $\\psi_N$ = {q4_mean:.4f}')

    # 95% CI shading
    ax.axvspan(q4_lo, q4_hi, alpha=0.15, color='red',
               label=f'95% CI: [{q4_lo:.4f}, {q4_hi:.4f}]')

    # Rug plot (individual observations)
    ax.plot(q4_locations, np.zeros_like(q4_locations) - 0.1,
            '|', color='k', ms=15, mew=1.5)

    ax.set_xlabel(r'$\psi_N$ of q = 4 surface', fontsize=13)
    ax.set_ylabel('Count', fontsize=13)
    ax.set_title(f'Distribution of q = 4 surface location\n'
                 f'({len(q4_locations)} perturbed equilibria, '
                 f'$\\sigma$ = {q4_std:.4f})', fontsize=14)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(bottom=-0.3)

    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'q4_histogram.png'), dpi=150, bbox_inches='tight')
    print(f"\nSaved: {FIG_DIR}/q4_histogram.png")
    plt.show()

else:
    print(f"WARNING: Only {len(q4_locations)} q=4 crossings found.")
    print("Cannot compute meaningful statistics. Check q-profiles above.")

## 12. Summary

Print the final results in a compact table for easy reference and
inclusion in reports.

In [ ]:
print("\n" + "="*70)
print("  RESULTS: q = 4 Surface Location Uncertainty for Shot 199749")
print("="*70)
print(f"  Equilibrium:       g199749.03350  (DIII-D)")
print(f"  Bouquet size:      {n_equils} perturbed equilibria")
print(f"  Uncertainty basis: Data-driven (raw Thomson/CER scatter)")
print(f"  GPR length scales: n={n_ls}, T={t_ls}, j={j_ls}")
print(f"  j_BS scale range:  {jBS_scale_range}")
print()

if len(q4_locations) >= 2:
    print(f"  q=4 locations found: {len(q4_locations)} / {n_equils + 1} equilibria")
    print()
    if q4_baseline:
        print(f"  Baseline q=4:      psi_N = {q4_baseline[0]:.4f}")
    print(f"  Mean q=4:          psi_N = {q4_mean:.4f}")
    print(f"  Std deviation:     {q4_std:.4f}")
    print(f"  95% CI:            psi_N in [{q4_lo:.4f}, {q4_hi:.4f}]")
    print()
    print(f"  The q=4 surface is located at psi_N = {q4_mean:.4f} +/- {q4_std:.4f}")
    print(f"  95% CI width:      {q4_hi - q4_lo:.4f} in psi_N")
else:
    print(f"  Insufficient q=4 crossings ({len(q4_locations)}) for statistics.")

print("="*70)

# ---- List all saved figures ----
print(f"\nSaved figures in {FIG_DIR}/:\n")
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith('.png'):
        fpath = os.path.join(FIG_DIR, f)
        fsize = os.path.getsize(fpath) / 1024
        print(f"  {f:40s}  ({fsize:.0f} kB)")

## 13. Save eqdsks

Save the final eqdsk files

In [ ]:
"""
Blend a perturbed PFile (reliable for psiN < 1) with a baseline IDA PFile
(reliable for psiN >= 1) to produce a single smooth profile across the full
grid.

The blend uses a sigmoid transition centred at psiN = 1 so that:
  - well inside  (psiN < 1 - half_width) → 100% perturbed
  - well outside (psiN > 1 + half_width) → 100% baseline
  - across the transition zone            → smooth sigmoidal mix

Usage
-----
    stitched = stitch_pfile(perturbed_pfile, ida_pfile)
"""



# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _sigmoid_blend(psin, centre=1.0, half_width=0.05):
    """
    Return a weight array w(psiN) in [0, 1].

    w ≈ 1  (use perturbed)  for psiN << centre
    w ≈ 0  (use baseline)   for psiN >> centre

    Parameters
    ----------
    psin       : array-like  normalised poloidal flux grid
    centre     : float       transition centre (default 1.0 = LCFS)
    half_width : float       e-folding half-width in psiN units
    """
    psin = np.asarray(psin, dtype=float)
    # sigmoid: 1 / (1 + exp((psin - centre) / scale))
    scale = half_width / np.log(9)   # w=0.9 at centre-half_width,
                                     # w=0.1 at centre+half_width
    w = 1.0 / (1.0 + np.exp((psin - centre) / scale))
    return w


def _interp(psin_src, data_src, psin_tgt):
    """Linear interpolation with flat extrapolation at boundaries."""
    f = interp1d(psin_src, data_src,
                 kind='linear', bounds_error=False,
                 fill_value=(data_src[0], data_src[-1]))
    return f(psin_tgt)


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def stitch_pfile(perturbed, baseline, *,
                 blend_centre=1.0,
                 blend_half_width=0.05,
                 profiles=('ne', 'te', 'ni', 'ti', 'nz1', 'omeg'),
                 recompute_pressure=True):
    """
    Blend a perturbed PFile with a baseline PFile across psiN ≈ 1.

    For each profile in *profiles* the output is::

        out = w * perturbed + (1 - w) * baseline

    where w is a sigmoid that equals 1 deep inside (psiN ≪ 1) and 0
    outside (psiN ≫ 1).

    The output grid is taken from *perturbed* (the grid that covers the
    full range needed by downstream tools).  The baseline is interpolated
    onto that grid before blending.

    Parameters
    ----------
    perturbed : PFile
        Perturbed equilibrium p-file (accurate inside psiN < 1).
    baseline : PFile
        IDA baseline p-file (accurate everywhere, used for psiN > 1 tail).
    blend_centre : float
        Centre of the sigmoid blend (default 1.0 = LCFS).
    blend_half_width : float
        Half-width of the transition in psiN units.  Over
        [centre - half_width, centre + half_width] the blend goes from
        ~90% perturbed to ~10% perturbed.  Default 0.05.
    profiles : tuple of str
        Which profile keys to stitch.  Profiles absent from either file
        are silently skipped.
    recompute_pressure : bool
        If True, call pf.compute_pressure() on the result to keep ptot
        consistent with the blended ne/te/ni/ti.

    Returns
    -------
    PFile
        New PFile with stitched profiles on the *perturbed* psiN grid.
    """
    out = PFile.new()

    # Copy ion species from perturbed (or fall back to baseline)
    nza = perturbed.ion_species or baseline.ion_species
    if nza is not None:
        out.set_ion_species(nza['N'], nza['Z'], nza['A'])

    for key in profiles:
        # Need the profile in at least one of the two files
        pert_entry = perturbed[key] if key in perturbed else None
        base_entry = baseline[key]  if key in baseline  else None

        if pert_entry is None and base_entry is None:
            continue   # not available in either — skip

        # Use whichever grid is available as the output grid
        if pert_entry is not None:
            psin_out  = pert_entry['psinorm']
            pert_data = pert_entry['data']
        else:
            # Perturbed file doesn't have this key; use baseline only
            psin_out  = base_entry['psinorm']
            pert_data = _interp(base_entry['psinorm'],
                                base_entry['data'], psin_out)

        if base_entry is not None:
            base_data = _interp(base_entry['psinorm'],
                                base_entry['data'], psin_out)
        else:
            # Baseline doesn't have this key; use perturbed only
            base_data = pert_data

        # Sigmoid weights
        w = _sigmoid_blend(psin_out,
                           centre=blend_centre,
                           half_width=blend_half_width)

        blended = w * pert_data + (1.0 - w) * base_data

        # Preserve units from whichever entry is available
        units = (pert_entry or base_entry).get('units', None)
        out.set_profile(key, psin_out, blended, units=units)

    if recompute_pressure:
        # Only possible if ne/te/ni/ti are all present
        required = {'ne', 'te', 'ni', 'ti'}
        if required.issubset(set(out.keys)):
            out.compute_pressure()

    return out

In [ ]:
# ---- Extract g- and p-files from all perturbed equilibria ----

EQ_DIR = 'gEQDSKs'
os.makedirs(EQ_DIR, exist_ok=True)

PF_DIR = 'pFiles'
os.makedirs(PF_DIR, exist_ok=True)

with h5py.File(f"{header}.h5", 'r') as hf:
    # Dynamically discover HDF5 layout
    # Layout is either: scan/{scan_val}/{count} or flat {count}
    if 'scan' in hf:
        scan_grp = hf['scan']
        scan_keys = list(scan_grp.keys())
        print(f"HDF5 layout: scan/ with keys {scan_keys}")
        equil_grp = scan_grp[scan_keys[0]]
    else:
        equil_grp = hf
        print("HDF5 layout: flat (no scan/ group)")

    eq_keys = sorted([k for k in equil_grp.keys() if k.isdigit()],
                     key=lambda k: int(k))

    print(f"Found {len(eq_keys)} equilibria in database (including baseline)")
    print()

    for eq_key in eq_keys:
        eq_grp = equil_grp[eq_key]
        eq_idx = int(eq_key)

        # Find eqdsk dataset by name ending in '.eqdsk'
        eqdsk_ds = [k for k in eq_grp.keys() if k.endswith('.eqdsk')]
        if eqdsk_ds:
            eqdsk_bytes = bytes(eq_grp[eqdsk_ds[0]][()])
            eq_data = read_eqdsk_from_bytes(eqdsk_bytes, read_geqdsk)
            eq_data.save(f'{EQ_DIR}/g{gpsave_header}-idx{eq_key}')
            
        else:
            print(f"  [{eq_key}]: no eqdsk dataset found")
            
        pfile_ds = [k for k in eq_grp.keys() if k.endswith('.pfile')]
        # if pfile_ds:
        #     pfile_bytes = bytes(eq_grp[pfile_ds[0]][()])
        #     pfile_data = PFile.from_bytes(pfile_bytes)
        #     pfile_data.save(f'{PF_DIR}/p{gpsave_header}-idx{eq_key}')
            
        if pfile_ds: # we need to stich with the original to get decent SOL profiles
            print(f'WARNING: stiching pFile from [{eq_key}] with original')
            pfile_bytes = bytes(eq_grp[pfile_ds[0]][()])
            pfile_data  = PFile.from_bytes(pfile_bytes)
            stitched    = stitch_pfile(pfile_data, ida_pfile, blend_half_width=0.01)
            stitched.save(f'{PF_DIR}/p{gpsave_header}-idx{eq_key}')
            
        else:
            print(f"  [{eq_key}]: no pFile dataset found")


In [ ]:
# Check the pFiles
ida_pfile_psin = ida_pfile.psinorm_for('ne')  # p-file psiN grid (256 points, 0 to ~1.05)

print(f"p-file grid: {len(ida_pfile_psin)} points, psiN = [{ida_pfile_psin[0]:.4f}, {ida_pfile_psin[-1]:.4f}]")
print(f"ne(0) = {ida_pfile.ne[0]:.4f} x 10^20/m^3")
print(f"te(0) = {ida_pfile.te[0]:.4f} keV")
print(f"ti(0) = {ida_pfile.ti[0]:.4f} keV")

# ---- Load fitted profiles from  new p-file ----
pf_new = read_pfile(f'pFiles/p{gpsave_header}-idx10')
pf_new_psin = pf_new.psinorm_for('ne')  # p-file psiN grid (256 points, 0 to ~1.05)

print(f"p-file grid: {len(pf_new_psin)} points, psiN = [{pf_new_psin[0]:.4f}, {pf_new_psin[-1]:.4f}]")
print(f"ne(0) = {pf_new.ne[0]:.4f} x 10^20/m^3")
print(f"te(0) = {pf_new.te[0]:.4f} keV")
print(f"ti(0) = {pf_new.ti[0]:.4f} keV")

plt.plot(ida_pfile_psin , ida_pfile.ne, c='k', label='orig')
plt.plot(pf_new_psin, pf_new.ne, c='r', label='new')
plt.xlabel('psin')
plt.ylabel('pFile ne')
plt.legend()
